# 1. Data quality and cleaning

This notebook loads the synthetic UCI AI4I 2020 snapshot. The official CSV stays unchanged. Missing values, exact duplicate rows, duplicate `uid` values, invalid records, and IQR outliers are audited before a canonical cleaned CSV is written. Official records are never imputed or removed; outliers are flagged only.

Run Jupyter from the repository root with `src` on `PYTHONPATH`.

In [ ]:
from pathlib import Path
import json
import pandas as pd

from industrial_analyzer.data_loader import read_ai4i_csv
from industrial_analyzer.cleaning import clean_official_ai4i_data

REPOSITORY_ROOT = Path.cwd()
assert (REPOSITORY_ROOT / "src/industrial_analyzer").is_dir(), "Start Jupyter from the repository root."
source_path = REPOSITORY_ROOT / "data/raw/ai4i2020.csv"
source = read_ai4i_csv(source_path)
print(f"Source shape: {source.shape}")
print(f"Total missing cells: {int(source.isna().sum().sum())}")
print(f"Exact duplicate rows: {int(source.duplicated().sum())}")
print(f"Duplicate uid records: {int(source['uid'].duplicated(keep=False).sum())}")

In [ ]:
cleaned = clean_official_ai4i_data()
report_path = REPOSITORY_ROOT / "outputs/reports/data_quality.json"
report = json.loads(report_path.read_text(encoding="utf-8"))
print(f"Cleaning status: {report['status']}")
print(f"Canonical shape: {cleaned.shape}")
print(f"Cleaned CSV: data/processed/ai4i_cleaned.csv")
print(f"Quality report: outputs/reports/data_quality.json")

In [ ]:
display(pd.DataFrame({
    "missing_values": report["missing_values_by_column"],
}))
display(pd.DataFrame({
    "field": list(report["outliers"]["fields"]),
    "flagged_readings": [item["flagged_count"] for item in report["outliers"]["fields"].values()],
}))

The separate dirty-data exercise lives under `data/demo/`. It demonstrates only the documented edits in its manifest; it does not alter the official data or the canonical cleaned file.

In [ ]:
demo_report_path = REPOSITORY_ROOT / "outputs/reports/dirty_demo_quality.json"
if demo_report_path.exists():
    demo_report = json.loads(demo_report_path.read_text(encoding="utf-8"))
    display(demo_report)
else:
    print("Run scripts/run_cleaning.py to generate the separate dirty-data demonstration.")